In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# Загрузка и подготовка данных
# Чтение CSV-файла
df = pd.read_csv("clean.csv", sep=",", decimal=".")

# Целевой параметр y
target = "Пtм в печи финиш, сред"

# Отделение входных признаков X от целевого параметра y
X = df.drop(columns=[target])
y = df[target]

# Разделение данных
# 90% данных — обучение, 10% — итоговая проверка
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10, # при 5% - r2 44%
    random_state=42
)

# Создание конвейера обработки данных и обучения модели
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", ExtraTreesRegressor(
        random_state=42,
        n_jobs=1
    ))
])

# Параметры для подбора
param_distributions = { # Не пробовал менять
    "model__n_estimators": [200, 400, 600],
    "model__max_features": [0.5, 0.7, 0.9, 1.0],
    "model__max_depth": [None, 15, 25, 40],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4, 8]
}

# Кросс-валидация
cv = KFold(n_splits=3, # попробовать увеличить (при 4 и 5% тестовой проверки доходит до r2 45%)
           shuffle=True, 
           random_state=42)

# Автоматический подбор параметров
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True
)

# Обучение модели с автоматическим подбором параметров
search.fit(X_train, y_train)

# Выбор лучшей модели
model = search.best_estimator_

# Оценка качества модели

# Предсказание y на тестовых данных
pred = model.predict(X_test)

# Рассчет основных метрик качества
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

# R² и MAE
print(f"R²:  {r2:.4f}")
print(f"MAE: {mae:.4f}")


# Важность признаков

# Перемешивание значений каждого признака по отдельности
# и измерение ухудшения R² модели
#result = permutation_importance(
#    model,
#    X_test,
#    y_test,
#    n_repeats=5,
#    random_state=42,
#    scoring="r2",
#    n_jobs=-1
#)

# Сортировка
#importance = pd.Series(
#   result.importances_mean,
#   index=X_test.columns
#).sort_values(ascending=False)

# Вывод
#print("\nВажность признаков:")
#print(importance.to_string())


R²:  0.4484
MAE: 3.4762
